# Classification of Ancient Handwritten Tamil Characters on Palm Leaf Inscriptions
## Using Modified Adaptive Backpropagation Neural Network (MABPN) with GLCM Features

**Primary Reference:**  
> Poornima Devi. M and M. Sornam (2020), *"Classification of Ancient Handwritten Tamil Characters on Palm Leaf Inscription Using Modified Adaptive Backpropagation Neural Network with GLCM Features"*, **ACM Transactions on Asian and Low-Resource Language Information Processing (ACM TALLIP)**, Vol. 19, Iss. 6, Article 82.  
> DOI: [https://doi.org/10.1145/3406209](https://doi.org/10.1145/3406209)

---
### Proposed Methodology Overview:
1. **Preprocessing (Section 3):** Grayscale -> Median Filter -> Adaptive Mean Threshold -> Convex Hull Segmentation -> Resize to uniform 8x8.
2. **Proposed Model 1 (MABPN-BCV):** 64-D Binary Coded Value input, 65 input neurons (including bias), 55 hidden neurons, 5 output neurons (5-bit class encoding, Figure 12).
3. **Proposed Model 2 (MABPN-GLCM):** 6 GLCM statistical features (Contrast, Energy, Homogeneity, Correlation, Dissimilarity, ASM), 6 input neurons (+ bias), 20 hidden neurons, 5 output neurons.
4. **Weight Initialization:** Nguyen-Widrow algorithm (Equations 1-6).
5. **Activation Functions:** Input-to-Hidden: Sigmoid (Eq. 7); Hidden-to-Output: Shannon Wavelet (Eq. 8).
6. **Learning Algorithm:** Modified Adaptive Backpropagation with differential linear and non-linear errors (Equations 9-20).

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

if "." not in sys.path:
    sys.path.insert(0, ".")

import config
import preprocess
import features
import neural_networks
import models
import evaluate
import dataset

print("Reference Paper modules loaded successfully.")
print(f"Target Convergence Error: {config.CONVERGENCE_ERROR_TARGET}")

### 1. Preprocessing Pipeline (Section 3 & Figures 4-8)
Demonstrating Grayscale conversion (Figure 4), Median Filtering (Figure 5), Adaptive Mean Thresholding (Figure 6), and Contour-based Convex Hull bounding box segmentation on an authentic manuscript sample.

In [ ]:
manuscript_path = config.MANUSCRIPTS_DIR / "tolkappiyam_palm_leaf.jpg"
if manuscript_path.exists():
    stages = preprocess.preprocess_palm_leaf_page(manuscript_path)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(stages["original"])
    axes[0].set_title("Original Palm Leaf (Fig. 3)")
    axes[1].imshow(stages["grayscale"], cmap="gray")
    axes[1].set_title("Grayscale Image (Fig. 4)")
    axes[2].imshow(stages["median_filter"], cmap="gray")
    axes[2].set_title("Median Filter (Fig. 5)")
    axes[3].imshow(stages["adaptive_threshold"], cmap="gray")
    axes[3].set_title("Adaptive Mean Threshold (Fig. 6)")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Manuscript file not found at:", manuscript_path)

### 2. Dataset Verification and Stratified 70:30 Split (Section 4.1.1)
The reference paper uses 1,756 characters from the 'Kuzhanthai Pini Marunthu' palm-leaf manuscript (U.V. Swaminatha Iyer Library, Chennai) split 70% train (1,229) and 30% test (527).

In [ ]:
X_train_raw, X_test_raw, y_train, y_test, class_names, status = dataset.load_character_dataset(
    data_dir=config.DATA_DIR,
    test_size=config.TRAIN_TEST_SPLIT_RATIO,
    random_state=config.RANDOM_SEED,
    allow_test_fixture=True
)
print(f"Character Classes: {len(class_names)}")
print(f"Training set (70%): {len(X_train_raw)} samples")
print(f"Testing set (30%):  {len(X_test_raw)} samples")

### 3. Model 1: MABPN-BCV (Binary Coded Value Model)
- Input: 8x8 binary character image -> 64 binary values in {0, 1}.
- Architecture: 65 input neurons (including bias), 55 hidden neurons, 5 output neurons (5-bit class encoding, Figure 12).
- Activation: Sigmoid (hidden) and Shannon Wavelet (output).

In [ ]:
X_tr_bcv = features.extract_features_batch(X_train_raw, model_type="bcv")
X_te_bcv = features.extract_features_batch(X_test_raw, model_type="bcv")
print("BCV Feature Shape:", X_tr_bcv.shape)

model_bcv = models.MABPN_BCV(
    learning_rate=0.05,
    max_iter=130,
    tol=config.CONVERGENCE_ERROR_TARGET,
    random_state=42
)
model_bcv.fit(X_tr_bcv, y_train)
y_pred_bcv = model_bcv.predict(X_te_bcv)

metrics_bcv = evaluate.compute_paper_metrics(y_test, y_pred_bcv, class_names)
print(evaluate.format_evaluation_summary("MABPN-BCV", metrics_bcv, len(model_bcv.loss_history_), model_bcv.loss_history_[-1]))

### 4. Model 2: MABPN-GLCM (GLCM Statistical Features Model)
- Feature Extraction: Exactly 6 GLCM features (Equations 21-27):
  1. Contrast (Eq. 21)
  2. Energy (Eq. 22)
  3. Homogeneity (Eq. 23)
  4. Correlation (Eq. 24)
  5. Dissimilarity (Eq. 26)
  6. Angular Second Moment (ASM) (Eq. 27)
- Architecture: 6 input neurons (+ bias), 20 hidden neurons, 5 output neurons.

In [ ]:
X_tr_glcm = features.extract_features_batch(X_train_raw, model_type="glcm")
X_te_glcm = features.extract_features_batch(X_test_raw, model_type="glcm")
print("GLCM Feature Shape:", X_tr_glcm.shape)

model_glcm = models.MABPN_GLCM(
    learning_rate=0.05,
    max_iter=104,
    tol=config.CONVERGENCE_ERROR_TARGET,
    random_state=42
)
model_glcm.fit(X_tr_glcm, y_train)
y_pred_glcm = model_glcm.predict(X_te_glcm)

metrics_glcm = evaluate.compute_paper_metrics(y_test, y_pred_glcm, class_names)
print(evaluate.format_evaluation_summary("MABPN-GLCM", metrics_glcm, len(model_glcm.loss_history_), model_glcm.loss_history_[-1]))

### 5. Training Loss Curves and Convergence Analysis (Figure 9)
Plotting mean squared error convergence against the paper stopping threshold (error < 0.0001).

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(model_bcv.loss_history_, label="MABPN-BCV (65-55-5)", color="#e74c3c", lw=2)
plt.plot(model_glcm.loss_history_, label="MABPN-GLCM (6-20-5)", color="#2980b9", lw=2)
plt.axhline(config.CONVERGENCE_ERROR_TARGET, color="black", linestyle="--", label="Stopping Target (0.0001)")
plt.xlabel("Training Epochs")
plt.ylabel("Mean Squared Error")
plt.title("MABPN Error Convergence Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()